In [1]:
import mysql.connector

In [3]:
conn = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "12345",
    database= "bookstore",
    port = 3306
)

In [4]:
if conn.is_connected():
    print("connection successful")

connection successful


In [5]:
cursor = conn.cursor()

In [6]:
query = """with customer_feature as(
		select Customer_ID , Name  ,City , Country
        from customers 
     
),
total_order as (
	select c.Customer_ID ,
    count(o.Order_ID) as total_order
    from customers c 
    left join orders o
		on c.Customer_ID = o.Customer_Id
	group by c.Customer_ID
),
total_books as (
	select c.Customer_ID , sum(o.Quantity) as total_books
	from customers c 
    left join orders o 
		on c.Customer_ID  = o.Customer_Id
	group by c.Customer_ID
),
total_spending AS (
    SELECT 
        c.Customer_ID,
        COALESCE(SUM(o.Total_Amount), 0) AS total_spending
    FROM customers c
    LEFT JOIN orders o
        ON c.Customer_ID = o.Customer_Id
    GROUP BY c.Customer_ID

),
average_order as  (
	select c.Customer_ID , 	avg(o.Total_Amount) as average_order_value
    from customers c
    left join orders o
		on c.Customer_ID = o.Customer_Id
	group by c.Customer_ID
),
average_books as (
	select c.Customer_ID , avg(o.Quantity) as average_book_value
	from customers c
    left join orders o 
		on c.Customer_ID = o.Customer_Id
	group by c.Customer_ID
),
unique_books as (
	select c.Customer_ID , count(distinct b.Book_ID) as unique_books ,
    count(distinct b.Genre)  as unique_genre 
    from customers c
    left join orders o
		on c.Customer_ID = o.Customer_Id
	left join books b
		on o.Book_ID = b.BOOK_ID
	group by c.Customer_ID
),
order_date as  (
	select c.Customer_ID , min(o.Order_Date) as first_order_date,
    max(o.Order_Date) as last_order_date,
    DATEDIFF(MAX(o.Order_Date), MIN(o.Order_Date)) as customer_lifetime_days,
    DATEDIFF(CURDATE(), MAX(o.Order_Date)) as  Days_Since_Last_Order
    from customers c
    left join orders o
		on c.Customer_ID = o.Customer_Id
	group by c.Customer_ID
),
customer_spending_category as (
	SELECT c.Customer_ID,c.Name,
    COALESCE(SUM(o.Total_Amount), 0) AS Total_Spending,
    CASE
        WHEN COALESCE(SUM(o.Total_Amount), 0) >= 400 THEN 'Premium'
        WHEN COALESCE(SUM(o.Total_Amount), 0) >= 250 THEN 'Regular'
        ELSE 'Basic'
    END AS Customer_Category
	FROM customers c
	LEFT JOIN orders o
		ON c.Customer_ID = o.Customer_ID
	GROUP BY c.Customer_ID, c.Name
)


SELECT
    cf.Customer_ID, cf.Name, cf.City, cf.Country,
    tor.total_order,
    tb.total_books,
    ts.total_spending,
    ao.average_order_value,
    ab.average_book_value,
    ub.unique_books,
    ub.unique_genre,
    od.first_order_date,
    od.last_order_date,
    od.customer_lifetime_days,
    od.Days_Since_Last_Order,
    csc.Customer_Category
FROM customer_feature cf

LEFT JOIN total_order tor
    ON cf.Customer_ID = tor.Customer_ID

LEFT JOIN total_books tb
    ON cf.Customer_ID = tb.Customer_ID

LEFT JOIN total_spending ts
    ON cf.Customer_ID = ts.Customer_ID

LEFT JOIN average_order ao
    ON cf.Customer_ID = ao.Customer_ID

LEFT JOIN average_books ab
    ON cf.Customer_ID = ab.Customer_ID

LEFT JOIN unique_books ub
    ON cf.Customer_ID = ub.Customer_ID

LEFT JOIN order_date od
    ON cf.Customer_ID = od.Customer_ID

LEFT JOIN customer_spending_category csc
    ON cf.Customer_ID = csc.Customer_ID;"""

In [8]:
cursor.execute(query)

In [9]:
print(cursor)

MySQLCursor: with customer_feature as(
		select Custo..


In [10]:
import pandas as pd

In [11]:
df = pd.DataFrame(cursor.fetchall(), columns=[i[0] for i in cursor.description])

In [12]:
df

,Customer_ID,Name,City,Country,total_order,total_books,total_spending,average_order_value,average_book_value,unique_books,unique_genre,first_order_date,last_order_date,customer_lifetime_days,Days_Since_Last_Order,Customer_Category
0,1,Deborah Griffith,South Craigfort,Denmark,0,None,0.00,None,None,0,0,None,None,NaN,NaN,Basic
1,2,Crystal Clements,East Derekberg,Nicaragua,2,10,314.62,157.310000,5.0000,2,2,2023-02-23,2024-08-31,555.0,752.0,Regular
2,3,Susan Fuller,Austinbury,Equatorial Guinea,0,None,0.00,None,None,0,0,None,None,NaN,NaN,Basic
3,4,Jamie Ramirez,Dianamouth,Slovenia,0,None,0.00,None,None,0,0,None,None,NaN,NaN,Basic
4,5,Marcus Murphy,Smithbury,Guinea-Bissau,0,None,0.00,None,None,0,0,None,None,NaN,NaN,Basic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,496,Michelle Myers,Davidview,Netherlands Antilles,1,9,316.26,316.260000,9.0000,1,1,2023-11-17,2023-11-17,0.0,1040.0,Regular
496,497,Shane Klein,West Billymouth,Cote d'Ivoire,2,12,295.23,147.615000,6.0000,2,2,2023-04-04,2024-03-20,351.0,916.0,Regular
497,498,Brianna Fischer,Angelatown,Gibraltar,1,7,87.01,87.010000,7.0000,1,1,2024-12-07,2024-12-07,0.0,654.0,Basic
498,499,Melissa Curtis,Ashleyport,Monaco,1,10,231.60,231.600000,10.0000,1,1,2023-02-19,2023-02-19,0.0,1311.0,Basic


In [14]:
df.to_csv('customer_features.csv', index=False)

In [15]:
df1 = df.copy()

In [16]:
df1.shape

(500, 16)

In [17]:
df1.isnull().sum()

Customer_ID                 0
Name                        0
City                        0
Country                     0
total_order                 0
total_books               193
total_spending              0
average_order_value       193
average_book_value        193
unique_books                0
unique_genre                0
first_order_date          193
last_order_date           193
customer_lifetime_days    193
Days_Since_Last_Order     193
Customer_Category           0
dtype: int64

In [19]:
df1.describe()

,Customer_ID,total_order,unique_books,unique_genre,customer_lifetime_days,Days_Since_Last_Order
count,500.000000,500.000000,500.000000,500.000000,307.000000,307.000000
mean,250.500000,1.000000,0.998000,0.920000,129.964169,945.938111
std,144.481833,1.028648,1.021804,0.907329,184.285944,197.131619
min,1.000000,0.000000,0.000000,0.000000,0.000000,654.000000
25%,125.750000,0.000000,0.000000,0.000000,0.000000,782.500000
50%,250.500000,1.000000,1.000000,1.000000,0.000000,926.000000
75%,375.250000,2.000000,2.000000,1.000000,253.500000,1089.500000
max,500.000000,6.000000,6.000000,4.000000,689.000000,1379.000000
